ACTUACION ESPECIAL DE FISCALIZACION RENDICION DE CUENTAS CONTRACTUAL PLATAFORMA SIA OBSERVA VIGENCIA 2025

IMPORTACION DE LIBRERIAS

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import platform
import distro
import unicodedata

from datetime import datetime

print(f"Las librerías se cargaron correctamente.")
print(f"Versión de Pandas: {pd.__version__}")
print(f"Versión de Numpy: {np.__version__}")
print(f"Sistema Operativo: {platform.system()} {platform.release()}")
print(f"Distribución Linux: {distro.name()} {distro.version()} ({distro.codename()})")

Las librerías se cargaron correctamente.
Versión de Pandas: 3.0.0
Versión de Numpy: 2.4.2
Sistema Operativo: Linux 6.14.0-37-generic
Distribución Linux: Ubuntu 24.04 (noble)


CONFIGURACIONES DE VISUALIZACION

In [8]:
pd.options.display.float_format = "{:,.2f}".format
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", lambda x: "%.2f" % x)
pd.set_option("display.max_info_columns", 500)
plt.style.use("ggplot")
print("configuraciones aplicadas")

configuraciones aplicadas


GESTION DE RUTAS

In [9]:
BASE_DIR = os.path.dirname(os.getcwd())
RAW_DATA_PATH = os.path.join(BASE_DIR, "data", "raw")

PROCESSED_DATA_PATH = os.path.join(BASE_DIR, "data", "processed")

InformeBasico = os.path.join(RAW_DATA_PATH, "Informe_Basico.xlsx")
InformeExtendido = os.path.join(RAW_DATA_PATH, "Informe_Extendido.xlsx")

try:
    df_basico = pd.read_excel(InformeBasico)
    df_extendido = pd.read_excel(InformeExtendido)
    print(f"proyecto ubicado en:{BASE_DIR}")
    print(
        f"Se carga archivo Informe Basico contiene {len(df_basico)} filas e Informe Extendido contiene {len(df_extendido)} filas"
    )

except Exception as e:

    print(f"Error {e}")

proyecto ubicado en:/home/wilo/Escritorio/auditoria_sia
Se carga archivo Informe Basico contiene 14199 filas e Informe Extendido contiene 14138 filas


LIMPIEZA DE CABECERAS

In [12]:
def normalizar_cabeceras(df):
    def limpiar_texto(texto):
        if not isinstance(texto, str):
            return texto

        texto = texto.replace("Ñ", "NH").replace("ñ", "nh")
        texto = (
            unicodedata.normalize("NFKD", texto)
            .encode("ascii", "ignore")
            .decode("utf-8")
        )

        return (
            texto.replace("NH", "N").strip().upper().replace(" ", "_").replace(".", "_")
        )

    df.columns = [limpiar_texto(col) for col in df.columns]
    return df


df_basico = normalizar_cabeceras(df_basico)
df_extendido = normalizar_cabeceras(df_extendido)

print("PROCESANDO INFORME BASICO")
df_basico["VIGENCIA"] = pd.to_numeric(df_basico["VIGENCIA"], errors="coerce").astype(
    "Int64"
)

cols_fecha_basico = [
    "FECHA_SUSCRIPCION",
    "FECHA_ACTA_DE_INICIO",
    "FECHA_TERMINACION",
    "FECHA_CREACION",
    "FECHA_TERMINACION_AMPLIADA",
]

for col in cols_fecha_basico:
    df_basico[col] = pd.to_datetime(
        df_basico[col].astype(str).str.strip().str[:10],
        format="%Y/%m/%d",
        errors="coerce",
    )

print("PROCESANDO INFORME EXTENDIDO")

df_extendido["VIGENCIA"] = pd.to_numeric(
    df_extendido["VIGENCIA"], errors="coerce"
).astype("Int64")
df_extendido["APROPIACION_INICIAL"] = pd.to_numeric(
    df_extendido["APROPIACION_INICIAL"], errors="coerce"
)

cols_fecha_ext = [
    "FECHA_SUSCRIPCION",
    "FECHA_ACTA_DE_INICIO",
    "FECHA_TERMINACION",
    "FECHA_CREACION",
   
]

for col in cols_fecha_ext:
    df_extendido[col] = pd.to_datetime(
        df_extendido[col].astype(str).str.strip().str[:10],
        format="%Y/%m/%d",
        errors="coerce"
    )

print("\n PROCESAMIENTO EXITOSO")
print(f"Básico: {df_basico.shape} | Extendido: {df_extendido.shape}")

PROCESANDO INFORME BASICO
PROCESANDO INFORME EXTENDIDO

 PROCESAMIENTO EXITOSO
Básico: (14199, 22) | Extendido: (14138, 29)


ENCABEZADOS Y ESTRUCTURA

In [13]:
def inspeccionar_dataframe(df, nombre):
    print(f"\n{'='*20} {nombre} {'='*20}")
    print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")

    resumen = pd.DataFrame(
        {
            "Columna": df.columns,
            "Tipo": df.dtypes.values,
            "No Nulos": df.notnull().sum().values,
            "Nulos": df.isnull().sum().values,
            "Ejemplo": [df[c].iloc[0] if len(df) > 0 else "N/A" for c in df.columns],
        }
    )

    display(resumen)

    print(f"\nVista previa de datos:")
    display(df.head(3))


inspeccionar_dataframe(df_basico, "INFORME BÁSICO")
inspeccionar_dataframe(df_extendido, "INFORME EXTENDIDO")


==================== INFORME BÁSICO ====================
Filas: 14199 | Columnas: 22


,Columna,Tipo,No Nulos,Nulos,Ejemplo
0,TIPO_DE_ENTIDAD,str,14199,0,SOCIEDADES DE ECONOMIA MIXTA
1,NIT,int64,14199,0,901685734
2,ENTIDAD,str,14199,0,UMBRALED S.A.S E.S.P.
3,VIGENCIA,Int64,14126,73,2025
4,CODIGO_CONTRATO,object,14198,1,UMB-01-2025
5,VALOR_INICIAL_CONTRATO,float64,14199,0,17676000.00
6,ADICIONES,float64,14199,0,0.00
7,LIBERACIONES,float64,14199,0,0.00
8,VALOR_VIGENTE,float64,14199,0,17676000.00
9,FECHA_SUSCRIPCION,datetime64[s],0,14199,NaT



Vista previa de datos:


,TIPO_DE_ENTIDAD,NIT,ENTIDAD,VIGENCIA,CODIGO_CONTRATO,VALOR_INICIAL_CONTRATO,ADICIONES,LIBERACIONES,VALOR_VIGENTE,FECHA_SUSCRIPCION,FECHA_ACTA_DE_INICIO,FECHA_TERMINACION,TIEMPO_EJECUCION,MODALIDAD_CONTRATACION,CAUSAL_CONTRATO,TIPO_CONTRATO,FECHA_CREACION,FECHA_TERMINACION_AMPLIADA,NIT_1,NOMBRE,TIPO,ESTADO_CONTRATO
0,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-01-2025,17676000.00,0.00,0.00,17676000.00,NaT,NaT,NaT,364,CONTRATACIÓN DIRECTA,PRESTACIÓN DE SERVICIOS PROFESIONALES Y APOYO,Apoyo a la Gestión,NaT,NaT,24547202,MARIA LILIANA MONTES HOYOS,Contratista,Rendido
1,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-02-2025,8846800.00,0.00,0.00,8846800.00,NaT,NaT,NaT,364,CONTRATACIÓN DIRECTA,PRESTACIÓN DE SERVICIOS PROFESIONALES Y APOYO,Apoyo a la Gestión,NaT,NaT,900010878,GRUPO METRO Y CIA. LTDA,Contratista,Rendido
2,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-03-2025,22176000.00,0.00,0.00,22176000.00,NaT,NaT,NaT,363,CONTRATACIÓN DIRECTA,PRESTACIÓN DE SERVICIOS PROFESIONALES Y APOYO,Apoyo a la Gestión,NaT,NaT,1088290219,DIANA MARCELA RIOS AGUIRRE,Contratista,Rendido



==================== INFORME EXTENDIDO ====================
Filas: 14138 | Columnas: 29


,Columna,Tipo,No Nulos,Nulos,Ejemplo
0,TIPO_DE_ENTIDAD,str,14138,0,SOCIEDADES DE ECONOMIA MIXTA
1,NIT,int64,14138,0,901685734
2,ENTIDAD,str,14138,0,UMBRALED S.A.S E.S.P.
3,VIGENCIA,Int64,14069,69,2025
4,CODIGO_CONTRATO,object,14137,1,UMB-01-2025
5,OBJETO_CONTRATO,str,14138,0,ARRENDAMIENTO DE INMUEBLE CON DESTINACION COMERCIAL
6,FECHA_SUSCRIPCION,datetime64[s],0,14138,NaT
7,FECHA_ACTA_DE_INICIO,datetime64[s],0,14138,NaT
8,FECHA_TERMINACION,datetime64[s],0,14138,NaT
9,TIEMPO_EJECUCION,int64,14138,0,364



Vista previa de datos:


,TIPO_DE_ENTIDAD,NIT,ENTIDAD,VIGENCIA,CODIGO_CONTRATO,OBJETO_CONTRATO,FECHA_SUSCRIPCION,FECHA_ACTA_DE_INICIO,FECHA_TERMINACION,TIEMPO_EJECUCION,VALOR_INICIAL_CONTRATO,ADICIONES,LIBERACIONES,VALOR_VIGENTE,MODALIDAD_CONTRATACION,CAUSAL_CONTRATO,TIPO_CONTRATO,NIT_1,NOMBRE,TIPO,NIT_2,NOMBRE_1,TIPO_1,NOMBRE_DEL_RUBRO,APROPIACION_INICIAL,ORIGEN_RECURSOS,CDPS,RPS,FECHA_CREACION
0,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-01-2025,ARRENDAMIENTO DE INMUEBLE CON DESTINACION COMERCIAL,NaT,NaT,NaT,364,17676000.00,0.00,0.00,17676000.00,CONTRATACIÓN DIRECTA,PRESTACIÓN DE SERVICIOS PROFESIONALES Y APOYO,Apoyo a la Gestión,24547202,MARIA LILIANA MONTES HOYOS,Contratista,1088332330,JORGE WILLIAM GONZALEZ TAMAYO,Interno,SERVICIOS FINANCIEROS Y SERVICIOS CONEXOS; SERVICIOS INMOBILIARIOS; Y SERVICIOS DE ARRENDAMIENTO Y LEASING,68266800.00,Recursos Propios,20250001,20250001,NaT
1,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-02-2025,"CONTRATAR LA IMPLEMENTACION, PUESTA EN FUNCIONAMIENTO Y MANTENIMIENTO DEL SOFTWARE AIRE, BAJO LA MODALIDAD DE ARRENDAMIENTO, QUE CONTENGA LOS MODULOS DE PRESUPUESTO, TESORERIA Y BANCOS, VENTANILLAS DE ATENCION AL PUBLICO, NOMINA Y TALENTO HUMANO, ALMACEN E INVENTARIOS, CONTABILIDAD Y SERVICIOS PUBLICOS DE ENERGIA.",NaT,NaT,NaT,364,8846800.00,0.00,0.00,8846800.00,CONTRATACIÓN DIRECTA,PRESTACIÓN DE SERVICIOS PROFESIONALES Y APOYO,Apoyo a la Gestión,900010878,GRUPO METRO Y CIA. LTDA,Contratista,1088332330,JORGE WILLIAM GONZALEZ TAMAYO,Interno,PAQUETES DE SOFTWARE,9072000.00,Recursos Propios,20250003,20250003,NaT
2,SOCIEDADES DE ECONOMIA MIXTA,901685734,UMBRALED S.A.S E.S.P.,2025,UMB-03-2025,PRESTAR EL SERVICIO PROFESIONAL EN EL CARGO DE DIRECCION ADMINISTRATIVA Y FINANCIERA PARA LA SOCIEDAD UMBRALED S.A.S. E.S.P.,NaT,NaT,NaT,363,22176000.00,0.00,0.00,22176000.00,CONTRATACIÓN DIRECTA,PRESTACIÓN DE SERVICIOS PROFESIONALES Y APOYO,Apoyo a la Gestión,1088290219,DIANA MARCELA RIOS AGUIRRE,Contratista,1088332330,JORGE WILLIAM GONZALEZ TAMAYO,Interno,SERVICIOS PRESTADOS A LAS EMPRESAS Y SERVICIOS DE PRODUCCION,171903986.00,Recursos Propios,20250005,20250003,NaT


In [15]:
# --- CHEQUEO PARA EL BÁSICO ---
errores_basico = df_basico[df_basico["FECHA_TERMINACION"] < df_basico["FECHA_ACTA_DE_INICIO"]]
df_basico["DURACION_REAL"] = (df_basico["FECHA_TERMINACION"] - df_basico["FECHA_ACTA_DE_INICIO"]).dt.days

# --- CHEQUEO PARA EL EXTENDIDO ---
errores_ext = df_extendido[df_extendido["FECHA_TERMINACION"] < df_extendido["FECHA_ACTA_DE_INICIO"]]
df_extendido["DURACION_REAL"] = (df_extendido["FECHA_TERMINACION"] - df_extendido["FECHA_ACTA_DE_INICIO"]).dt.days

print(f"🚩 BÁSICO: {len(errores_basico)} errores encontrados.")
print(f"🚩 EXTENDIDO: {len(errores_ext)} errores encontrados.")

🚩 BÁSICO: 0 errores encontrados.
🚩 EXTENDIDO: 0 errores encontrados.
